# 04. DeepSurv - Deep Learning for Survival

Train and evaluate neural network survival model.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.config import PROCESSED_DATA_DIR
from src.utils.io_utils import load_parquet
from src.models.deepsurv_model import train_deepsurv_model, predict_risk_deepsurv
from src.evaluation.metrics import compute_concordance_index, stratify_by_risk
from src.evaluation.survival_plots import plot_kaplan_meier

%matplotlib inline

## 1. Load Preprocessed Data

In [ ]:
# Load train and test data
train_df = load_parquet(PROCESSED_DATA_DIR / 'train.parquet')
test_df = load_parquet(PROCESSED_DATA_DIR / 'test.parquet')

print(f"Train: {train_df.shape}, Test: {test_df.shape}")

In [ ]:
# Prepare features
feature_cols = [c for c in train_df.columns if c not in ['patient_id', 'OS_time', 'OS_status']]

X_train = train_df[feature_cols]
y_train = train_df[['OS_time', 'OS_status']]
X_test = test_df[feature_cols]
y_test = test_df[['OS_time', 'OS_status']]

# Create validation set
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

## 2. Train DeepSurv Model

In [ ]:
# Train model
model, trainer = train_deepsurv_model(
    X_train, y_train,
    X_val, y_val,
    hidden_dims=[64, 32, 16],
    num_epochs=50,
    batch_size=32
)

## 3. Evaluate Model

In [ ]:
# Predict risk scores
test_risk = predict_risk_deepsurv(trainer, X_test)

# Compute C-index
c_index = compute_concordance_index(
    event_times=y_test['OS_time'].values,
    predicted_risk=test_risk,
    event_observed=y_test['OS_status'].values
)

print(f"DeepSurv C-index: {c_index:.4f}")

## 4. Risk Stratification

In [ ]:
# Stratify by risk
risk_groups = stratify_by_risk(test_risk, n_groups=2)

# Plot KM curves
fig = plot_kaplan_meier(
    event_times=y_test['OS_time'].values,
    event_observed=y_test['OS_status'].values,
    groups=risk_groups,
    group_labels=['Low Risk', 'High Risk'],
    title='DeepSurv - Risk Stratification'
)
plt.show()

In [ ]:
# Risk score distribution
from src.evaluation.survival_plots import plot_risk_distribution

fig = plot_risk_distribution(
    test_risk,
    event_observed=y_test['OS_status'].values,
    title='DeepSurv Risk Scores'
)
plt.show()

## Summary

- Trained DeepSurv neural network
- Evaluated performance with C-index
- Performed risk stratification
- Next: Model interpretability with SHAP